# Does my feature carry information?

[`strategy_test`](../functions_fin/strategy_test.md) with `method="shift"` keeps
the prices as they are and rotates your feature inputs in time by one random
offset. The feature keeps its own behaviour but loses its alignment with the
market. If the strategy does as well with the feature misaligned, the feature
carries no information.

The data is a mean-reverting market, where a price z-score carries real
information, and a noise feature that does not.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import RollingZscore, BandSignal, BacktestOHLCTarget, strategy_test

T, SUB, PPY = 2000, 6, 24 * 365
rng = np.random.default_rng(8)
x = np.zeros(T * SUB)                    # log price pulled back to its mean
e = rng.standard_normal(T * SUB) * 0.01 / np.sqrt(SUB)
for i in range(1, T * SUB):
    x[i] = x[i - 1] * (1 - 0.02 / SUB) + e[i]
p = (100 * np.exp(x)).reshape(T, SUB)
c = p[:, -1]
o = np.r_[p[0, 0], c[:-1]]
h, l = np.maximum(p.max(axis=1), o), np.minimum(p.min(axis=1), o)

zscore = RollingZscore(50)(np.log(c))
noise = RollingZscore(50)(np.convolve(rng.standard_normal(T), np.ones(50) / 50, mode="same"))

## Test each feature

The strategy trades a band on whatever feature it is given: short above 1.5,
long below -1.5, flat again at 0. The feature is the fifth input, so
`columns=(4,)` names it as the one to rotate.

In [ ]:
engine = BacktestOHLCTarget(taker_fee=5e-4)


def band_on(o, h, l, c, feature):
    target = np.nan_to_num(BandSignal(entry=1.5, exit=0.0)(feature))
    return engine(target, o, h, l, c)


results = {}
for name, feature in [("price z-score", zscore), ("noise", noise)]:
    results[name] = strategy_test(band_on, o, h, l, c, feature, n=499, method="shift",
                                  columns=(4,), periods_per_year=PPY)
    print(f"{name:14s} Sharpe {results[name].observed:6.2f}, p = {results[name].p_value:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, (name, res) in zip(axes, results.items()):
    ax.hist(res.null, bins=40, color="lightsteelblue", label="feature shifted")
    ax.axvline(res.observed, color="crimson", lw=2, label=f"aligned, p = {res.p_value:.3f}")
    ax.set_title(name, fontsize=10); ax.set_xlabel("Sharpe ratio"); ax.legend(fontsize=8)
plt.tight_layout()

Aligned with the market, the z-score gives a Sharpe ratio no shifted copy
reaches. The noise feature does no better aligned than shifted.

## Adapt it

- Pass every input your strategy needs and list the feature positions in
  `columns`. Several features rotate together, so their relations to each other
  are kept and only their alignment with the market is broken.
- `min_shift` sets the smallest rotation (a tenth of the series by default).
  Raise it for features with long memory, so a shifted copy is not close to
  the original.
- To test the market structure itself rather than one feature, use the default
  `method="bars"`; see
  [I tuned my parameters. Is the result real?](21-is-the-tuned-result-real.ipynb).